# QuantumQUBO Agent — Statistics & Proof of Results

This notebook reproduces all numbers reported in the paper:
- Table 1: Ablation results (Accuracy, Tokens)
- Table 2: Error type distribution (Code Error %, Formulation Error %)

**Seed**: 42 | **Benchmarks**: 100

In [1]:
import csv
import pandas as pd
from collections import Counter
from pathlib import Path

ROOT = Path(".")  # QuantumQUBO Agent directory

FULL_PIPELINE   = ROOT / "results/benchmark_suite_completed.csv"
NO_RETRY        = ROOT / "results/no_retries_full_pipeline_42.csv"
NO_JUDGE        = ROOT / "scripts/ablation_scripts/No Judge ablation study/results/no_judge_no_retries_42.csv"
DIRECT_METHOD   = ROOT / "scripts/ablation_scripts/Direct Method/results/ablation_direct_method_qwen_qwen3_coder_next_42.csv"

def load(path):
    """Load CSV, deduplicate by benchmark name (keep last row)."""
    rows = list(csv.DictReader(open(path, encoding='utf-8')))
    seen = {}
    for r in rows:
        seen[r['benchmark']] = r
    return list(seen.values())

fp   = load(FULL_PIPELINE)
nr   = load(NO_RETRY)
nj   = load(NO_JUDGE)
dm   = load(DIRECT_METHOD)

print(f"Full Pipeline   : {len(fp)} benchmarks")
print(f"No Retry        : {len(nr)} benchmarks")
print(f"No Judge        : {len(nj)} benchmarks")
print(f"Direct Method   : {len(dm)} benchmarks")

Full Pipeline   : 100 benchmarks
No Retry        : 100 benchmarks
No Judge        : 100 benchmarks
Direct Method   : 99 benchmarks


## Table 1 — Ablation Results (Accuracy & Avg Tokens)

In [2]:
def accuracy(rows):
    n = len(rows)
    success = sum(1 for r in rows if r['status'] == 'success')
    return success, n, round(100 * success / n, 1)

def avg_tokens(rows):
    tokens = [int(r['tokens_used']) for r in rows
              if r.get('tokens_used', '').strip().lstrip('-').isdigit()
              and int(r['tokens_used']) > 0]
    return round(sum(tokens) / len(tokens) / 1000, 1) if tokens else 0

methods = {
    'Full Pipeline (QuantumQUBO Agent)': fp,
    'w/o Iterative Feedback':            nr,
    'w/o Judge + w/o Iterative Feedback': nj,
    'Direct Method':                     dm,
}

print(f"{'Method':<42} {'Success':>8} {'Total':>6} {'Acc (%)':>8} {'Avg Tokens (k)':>15}")
print("-" * 82)
for name, rows in methods.items():
    s, n, acc = accuracy(rows)
    tok = avg_tokens(rows)
    print(f"{name:<42} {s:>8} {n:>6} {acc:>8} {tok:>15}")

Method                                      Success  Total  Acc (%)  Avg Tokens (k)
----------------------------------------------------------------------------------
Full Pipeline (QuantumQUBO Agent)                68    100     68.0            48.1
w/o Iterative Feedback                           35    100     35.0            24.7
w/o Judge + w/o Iterative Feedback               36    100     36.0            22.1
Direct Method                                    46     99     46.5             3.2


## Table 2 — Error Type Distribution

### Full Pipeline: classify by which agent exhausted retries

In [3]:
coding_enc = coding_fail = form_enc = form_fail = plan_enc = plan_fail = 0

for r in fp:
    pr = int(r.get('planner_retries') or 0)
    fr = int(r.get('formulizer_retries') or 0)
    cr = int(r.get('coder_retries') or 0)
    s  = r['status'] == 'success'

    if cr > 0:
        coding_enc += 1
        if not s: coding_fail += 1
    elif fr > 0:
        form_enc += 1
        if not s: form_fail += 1
    elif pr > 0:
        plan_enc += 1
        if not s: plan_fail += 1

total_errors = coding_enc + form_enc + plan_enc

print("=== Full Pipeline Error Encounters (success + failed) ===")
print(f"Coding errors     : {coding_enc:>3}  (recovered: {coding_enc - coding_fail}, failed: {coding_fail})")
print(f"Formulation errors: {form_enc:>3}  (recovered: {form_enc - form_fail}, failed: {form_fail})")
print(f"Planning errors   : {plan_enc:>3}  (recovered: {plan_enc - plan_fail}, failed: {plan_fail})")
print(f"Total errors      : {total_errors}")
print()
print("=== As % of total errors ===")
print(f"Coding errors     : {100*coding_enc/total_errors:.1f}%")
print(f"Formulation errors: {100*form_enc/total_errors:.1f}%")
print(f"Planning errors   : {100*plan_enc/total_errors:.1f}%")

=== Full Pipeline Error Encounters (success + failed) ===
Coding errors     :  46  (recovered: 18, failed: 28)
Formulation errors:   9  (recovered: 6, failed: 3)
Planning errors   :   3  (recovered: 2, failed: 1)
Total errors      : 58

=== As % of total errors ===
Coding errors     : 79.3%
Formulation errors: 15.5%
Planning errors   : 5.2%


### Direct Method: classify by failure_modes column

In [4]:
failed_dm = [r for r in dm if r['status'] == 'failed']
modes = Counter(r.get('failure_modes', 'other') or 'other' for r in failed_dm)
total_dm = len(failed_dm)

print("=== Direct Method Failures ===")
print(f"Total failures: {total_dm} / {len(dm)}")
print()
for mode, count in modes.most_common():
    label = mode if mode else 'other'
    print(f"  {label:<25}: {count:>3}  ({100*count/total_dm:.1f}%)")

=== Direct Method Failures ===
Total failures: 53 / 99

  formulation_error        :  35  (66.0%)
  coding_error             :  14  (26.4%)
  other                    :   4  (7.5%)


## Summary Table (Paper Numbers)

In [5]:
summary = {
    'QuantumQUBO Agent': {'Code Error (%)': 79.3, 'Form. Error (%)': 15.5, 'Other (%)': 5.2},
    'Direct Method':     {'Code Error (%)': 26.4, 'Form. Error (%)': 66.0, 'Other (%)': 7.5},
}

df = pd.DataFrame(summary).T
print(df.to_string())
print()
print("Row sums (should be ~100%):")
print(df.sum(axis=1))

                   Code Error (%)  Form. Error (%)  Other (%)
QuantumQUBO Agent            79.3             15.5        5.2
Direct Method                26.4             66.0        7.5

Row sums (should be ~100%):
QuantumQUBO Agent    100.0
Direct Method         99.9
dtype: float64
